# 01 — Multimodal Data Ingestion & Evidence Layer

This notebook converts the raw research PDFs into a structured, traceable evidence layer using **Docling**.

The ingestion layer preserves:
- text and section hierarchy
- lists and formulas
- tables with structured/markdown representations
- figures as actual extracted images
- figure/table captions and nearby explanatory text
- page numbers and bounding boxes
- stable document and element IDs

Downstream contract:
`document → evidence elements → chunks → retrieval → grounded answer → original PDF evidence`

In [1]:
from pathlib import Path
import json
import hashlib
import re
from collections import Counter

import docling
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

print('Docling version:', getattr(docling, '__version__', 'unknown'))

c:\Users\MD RAJA\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Docling version: 2.132.0


In [2]:
# Project paths
PDF_FOLDER = Path('../data/raw_papers')
PROCESSED_FOLDER = Path('../data/processed_papers')
IMAGE_FOLDER = PROCESSED_FOLDER / 'images'

PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)
IMAGE_FOLDER.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(PDF_FOLDER.glob('*.pdf'))

print('PDF folder:', PDF_FOLDER.resolve())
print('PDFs found:', len(pdf_files))
for pdf in pdf_files:
    print(' -', pdf.name)

PDF folder: C:\Users\MD RAJA\Quant_Research_Intelligence_System\data\raw_papers
PDFs found: 11
 - 2508.15237v2 (1).pdf
 - 2605.21504v1.pdf
 - 2605.23953v1.pdf
 - 2605.23959v1.pdf
 - 2605.25894v1.pdf
 - 2605.27848v1.pdf
 - 2605.27945v1.pdf
 - 2605.27977v1.pdf
 - 2605.28853v1.pdf
 - 2605.29413v1.pdf
 - ssrn-5231379.pdf


In [3]:
# Docling configuration
# We generate individual figure images because figures are part of the evidence layer.
# Full page images are not generated to avoid unnecessary storage.

pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True
pipeline_options.generate_page_images = False

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

print('Docling converter ready.')

Docling converter ready.


## Evidence representation

Every extracted element keeps its source location and identity. Figures and tables are **first-class evidence**, not just text descriptions.

For a figure, the JSON keeps:
`figure_id + page + section_path + caption + image_path + surrounding_text + related_element_ids`.

For a table, it keeps:
`table_id + page + section_path + caption + table_markdown + table_data + related_element_ids`.

In [4]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def stable_id(prefix: str, *parts) -> str:
    raw = '||'.join(str(p) for p in parts)
    digest = hashlib.sha1(raw.encode('utf-8')).hexdigest()[:12]
    return f'{prefix}_{digest}'


def get_page(item):
    try:
        return item.prov[0].page_no
    except Exception:
        return None


def get_bbox(item):
    try:
        box = item.prov[0].bbox
        return {'l': box.l, 't': box.t, 'r': box.r, 'b': box.b}
    except Exception:
        return None


def get_text(item):
    try:
        return item.text or ''
    except Exception:
        return ''


def get_formula_raw(item):
    # Docling formula items can have useful content in `orig` while `text` is empty.
    for attr in ('orig', 'text'):
        try:
            value = getattr(item, attr, None)
            if value:
                return str(value)
        except Exception:
            pass
    return ''


def normalize_type(item):
    name = type(item).__name__.lower()
    mapping = {
        'sectionheaderitem': 'section',
        'textitem': 'text',
        'listitem': 'list_item',
        'formulaitem': 'formula',
        'pictureitem': 'figure',
        'tableitem': 'table',
    }
    return mapping.get(name, name.replace('item', ''))

In [5]:
def safe_export_table(table_item, doc):
    """Export table evidence without failing the whole document."""
    table_markdown = None
    table_data = None

    try:
        table_markdown = table_item.export_to_markdown(doc)
    except Exception:
        pass

    try:
        df = table_item.export_to_dataframe(doc)
        columns = [str(c) for c in df.columns]
        table_data = {
            'columns': columns,
            'rows': df.fillna('').astype(str).to_dict(orient='records'),
            'has_duplicate_columns': len(columns) != len(set(columns)),
        }
    except Exception:
        pass

    return table_markdown, table_data

In [8]:
def extract_structured_document(pdf_path: Path):
    """Convert one PDF into the multimodal evidence representation."""

    result = converter.convert(pdf_path)
    doc = result.document

    source_hash = sha256_file(pdf_path)
    document_id = stable_id("doc", pdf_path.name, source_hash)

    document_image_folder = IMAGE_FOLDER / document_id
    document_image_folder.mkdir(parents=True, exist_ok=True)

    elements = []
    section_stack = []
    figure_counter = 0
    table_counter = 0
    formula_counter = 0

    # ------------------------------------------------------------
    # PASS 1: native Docling elements in document order
    # ------------------------------------------------------------
    for order, item_data in enumerate(doc.iterate_items(), start=1):

        # Current Docling version returns: (item, level)
        if isinstance(item_data, tuple):
            item, level = item_data
        else:
            item = item_data
            level = 1

        element_type = normalize_type(item)
        page = get_page(item)
        bbox = get_bbox(item)
        text = get_text(item)

        raw = (
            get_formula_raw(item)
            if element_type == "formula"
            else text
        )

        # --------------------------------------------------------
        # Section hierarchy
        # --------------------------------------------------------
        if element_type == "section":
            try:
                level = max(1, min(int(level), 20))
            except Exception:
                level = 1

            section_stack = section_stack[:level - 1]

            if text.strip():
                section_stack.append(text.strip())

        # --------------------------------------------------------
        # Base element metadata
        # --------------------------------------------------------
        element = {
            "element_id": stable_id(
                "elem",
                document_id,
                order,
                element_type,
                page,
                raw[:100],
            ),
            "document_id": document_id,
            "order": order,
            "type": element_type,
            "raw_type": type(item).__name__,
            "page": page,
            "bbox": bbox,
            "section_path": list(section_stack),
            "text": text,
            "raw": raw,
            "image_path": None,
            "caption": None,
            "surrounding_text": None,
            "related_element_ids": [],
            "parent_element_id": None,
            "figure_id": None,
            "table_id": None,
            "formula_id": None,
            "table_markdown": None,
            "table_data": None,
        }

        # --------------------------------------------------------
        # FIGURE
        # --------------------------------------------------------
        if element_type == "figure":
            figure_counter += 1
            element["figure_id"] = f"figure_{figure_counter:04d}"

            try:
                image = item.get_image(doc)

                if image is not None:
                    image_file = (
                        document_image_folder
                        / f"{element['figure_id']}.png"
                    )

                    image.save(image_file)

                    element["image_path"] = str(
                        image_file.relative_to(PROCESSED_FOLDER)
                    )

            except Exception as exc:
                element["image_error"] = str(exc)

        # --------------------------------------------------------
        # TABLE
        # --------------------------------------------------------
        elif element_type == "table":
            table_counter += 1
            element["table_id"] = f"table_{table_counter:04d}"

            (
                element["table_markdown"],
                element["table_data"],
            ) = safe_export_table(item, doc)

        # --------------------------------------------------------
        # FORMULA
        # --------------------------------------------------------
        elif element_type == "formula":
            formula_counter += 1
            element["formula_id"] = f"formula_{formula_counter:04d}"

        elements.append(element)

    # ------------------------------------------------------------
    # PASS 2: conservative local figure/table relationships
    # ------------------------------------------------------------

    caption_pattern = re.compile(
        r"^\s*(figure|fig\.?|table)\s*\d+",
        re.IGNORECASE,
    )

    for i, element in enumerate(elements):

        if element["type"] not in {"figure", "table"}:
            continue

        nearby = []

        for j in range(
            max(0, i - 3),
            min(len(elements), i + 4),
        ):
            if j == i:
                continue

            candidate = elements[j]

            if (
                candidate["page"] == element["page"]
                and candidate["type"] in {"text", "list_item"}
                and candidate["text"].strip()
            ):
                nearby.append(candidate)

        # --------------------------------------------------------
        # Caption detection
        # --------------------------------------------------------
        caption_candidate = next(
            (
                x
                for x in nearby
                if caption_pattern.match(x["text"].strip())
            ),
            None,
        )

        if caption_candidate:
            element["caption"] = caption_candidate["text"].strip()

            element["related_element_ids"].append(
                caption_candidate["element_id"]
            )

        # --------------------------------------------------------
        # Surrounding context
        # --------------------------------------------------------
        context_candidates = [
            x
            for x in nearby
            if (
                not caption_candidate
                or x["element_id"]
                != caption_candidate["element_id"]
            )
        ][:2]

        if context_candidates:
            element["surrounding_text"] = "\n".join(
                x["text"].strip()
                for x in context_candidates
            )

            element["related_element_ids"].extend(
                x["element_id"]
                for x in context_candidates
            )

    # ------------------------------------------------------------
    # Make local relationships bidirectional
    # ------------------------------------------------------------

    by_id = {
        e["element_id"]: e
        for e in elements
    }

    for element in elements:

        for related_id in element["related_element_ids"]:

            related = by_id.get(related_id)

            if (
                related
                and element["element_id"]
                not in related["related_element_ids"]
            ):
                related["related_element_ids"].append(
                    element["element_id"]
                )

    # ------------------------------------------------------------
    # Document metadata
    # ------------------------------------------------------------

    pages = [
        e["page"]
        for e in elements
        if e["page"] is not None
    ]

    title_candidate = next(
        (
            e["text"].strip()
            for e in elements
            if (
                e["type"] == "section"
                and e["page"] == 1
                and e["text"].strip()
            )
        ),
        pdf_path.stem,
    )

    return {
        "document_id": document_id,
        "source_file": pdf_path.name,
        "source_path": str(pdf_path),
        "source_sha256": source_hash,
        "parser": "Docling",
        "docling_version": getattr(
            docling,
            "__version__",
            "unknown",
        ),
        "page_count": max(pages) if pages else 0,
        "title_candidate": title_candidate,
        "element_counts": dict(
            Counter(
                e["type"]
                for e in elements
            )
        ),
        "elements": elements,
    }

## Test one PDF first

Run this cell before processing the complete corpus. Verify that figures have actual image paths and tables have extracted content.

In [9]:
test_pdf = pdf_files[0]
print('Testing:', test_pdf.name)

test_document = extract_structured_document(test_pdf)

print('Pages:', test_document['page_count'])
print('Elements:', len(test_document['elements']))
print('Counts:', test_document['element_counts'])

figures = [e for e in test_document['elements'] if e['type'] == 'figure']
tables = [e for e in test_document['elements'] if e['type'] == 'table']

print('Figures:', len(figures))
print('Figures with images:', sum(bool(e.get('image_path')) for e in figures))
print('Tables:', len(tables))
print('Tables with content:', sum(bool(e.get('table_markdown') or e.get('table_data')) for e in tables))

Testing: 2508.15237v2 (1).pdf
Pages: 30
Elements: 291
Counts: {'section': 17, 'text': 150, 'formula': 87, 'list_item': 29, 'table': 3, 'figure': 5}
Figures: 5
Figures with images: 5
Tables: 3
Tables with content: 3


C:\Users\MD RAJA\AppData\Local\Temp\ipykernel_21116\3905666683.py:16: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  'rows': df.fillna('').astype(str).to_dict(orient='records'),


In [10]:
# Inspect figure/table evidence metadata from the test PDF.
for element in test_document['elements']:
    if element['type'] not in {'figure', 'table'}:
        continue

    print('\n' + '=' * 80)
    print('TYPE:', element['type'])
    print('ID:', element.get('figure_id') or element.get('table_id'))
    print('Page:', element['page'])
    print('Section:', ' > '.join(element['section_path']))
    print('Caption:', element['caption'])
    print('Image:', element['image_path'])
    print('Related elements:', element['related_element_ids'])

    if element['type'] == 'figure':
        print('Surrounding text:', element['surrounding_text'])
    else:
        print('Table markdown preview:')
        print((element['table_markdown'] or '')[:1200])


TYPE: table
ID: table_0001
Page: 12
Section: 3 Numerical examples
Caption: Table 1: Examples of the stochastic volatility processes with their corresponding SDEs. Here, vt is the volatility with initial value v0 > 0, α = 1 2 - H ∈ (0, 1/2) and κ, θ, σ, η ∈ R.
Image: None
Related elements: ['elem_b6e22446cf0e', 'elem_371d0f1cdef2', 'elem_2ea9eb9eb924']
Table markdown preview:
| Volatility Process/Model   | Stochastic Differential Equations                          |
|----------------------------|------------------------------------------------------------|
| OU [25]                    | dvt = κ(θ - vt)dt + ηdWt                                   |
| mGBM [19]                  | dvt = κ(θ - vt)dt + (η + σvt)dWt                           |
| rHeston                    | vt = v0 + ∫ t 0 K(t - s)f (vs)ds + ∫ t 0 K(t - s)g(vs)dWs, |
| [12]                       | K(t - s) = (t-s)-α Γ(1-α) , f(vs) = κ(θ -vs), g(vs) = σ√vs |
| rBergomi [3]               | vt = v0 exp (η t 0 (t - s)-αdWs)      

In [11]:
# Save the test document immediately.
single_output = PROCESSED_FOLDER / f"{test_document['document_id']}.json"

with open(single_output, 'w', encoding='utf-8') as f:
    json.dump(test_document, f, ensure_ascii=False, indent=2)

print('Saved:', single_output)

Saved: ..\data\processed_papers\doc_e37d68152c78.json


In [12]:
import json

json_file = PROCESSED_FOLDER / f"{test_document['document_id']}.json"

with open(json_file, "r", encoding="utf-8") as f:
    saved_doc = json.load(f)

tables = [
    e for e in saved_doc["elements"]
    if e["type"] == "table"
]

print("JSON file:", json_file)
print("Tables in JSON:", len(tables))

for table in tables:
    print("\n" + "=" * 80)
    print("TABLE:", table["table_id"])
    print("PAGE:", table["page"])
    print("SECTION:", table["section_path"])
    print("CAPTION:", table["caption"])
    print("\nMARKDOWN:")
    print(table["table_markdown"])
    print("\nTABLE DATA:")
    print(table["table_data"])

JSON file: ..\data\processed_papers\doc_e37d68152c78.json
Tables in JSON: 3

TABLE: table_0001
PAGE: 12
SECTION: ['3 Numerical examples']
CAPTION: Table 1: Examples of the stochastic volatility processes with their corresponding SDEs. Here, vt is the volatility with initial value v0 > 0, α = 1 2 - H ∈ (0, 1/2) and κ, θ, σ, η ∈ R.

MARKDOWN:
| Volatility Process/Model   | Stochastic Differential Equations                          |
|----------------------------|------------------------------------------------------------|
| OU [25]                    | dvt = κ(θ - vt)dt + ηdWt                                   |
| mGBM [19]                  | dvt = κ(θ - vt)dt + (η + σvt)dWt                           |
| rHeston                    | vt = v0 + ∫ t 0 K(t - s)f (vs)ds + ∫ t 0 K(t - s)g(vs)dWs, |
| [12]                       | K(t - s) = (t-s)-α Γ(1-α) , f(vs) = κ(θ -vs), g(vs) = σ√vs |
| rBergomi [3]               | vt = v0 exp (η t 0 (t - s)-αdWs)                           |

TABLE DATA:


## Process the complete corpus safely

Each PDF is converted and saved independently. If processing is interrupted, already completed documents remain available. Existing JSON files are skipped.

In [13]:
saved_documents = []
failed_documents = []

for pdf_path in pdf_files:
    print('\n' + '=' * 70)
    print('Processing:', pdf_path.name)

    document_id = stable_id(
        'doc', pdf_path.name, sha256_file(pdf_path)
    )
    output_file = PROCESSED_FOLDER / f'{document_id}.json'

    if output_file.exists():
        print('Already processed → SKIPPING')
        with open(output_file, 'r', encoding='utf-8') as f:
            saved_documents.append(json.load(f))
        continue

    try:
        document = extract_structured_document(pdf_path)

        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(document, f, ensure_ascii=False, indent=2)

        saved_documents.append(document)
            
        print('SUCCESS')
        print('Pages:', document['page_count'])
        print('Elements:', len(document['elements']))
        print('Figures:', document['element_counts'].get('figure', 0))
        print('Tables:', document['element_counts'].get('table', 0))
        print('Saved:', output_file)

    except KeyboardInterrupt:
        print('\nProcessing interrupted by user.')
        print('Already completed PDFs remain safely saved.')
        break

    except Exception as exc:
        failed_documents.append({
            'source_file': pdf_path.name,
            'error': str(exc),
        })
        print('FAILED:', exc)

print('\n' + '=' * 70)
print('COMPLETED:', len(saved_documents))
print('FAILED:', len(failed_documents))


Processing: 2508.15237v2 (1).pdf
Already processed → SKIPPING

Processing: 2605.21504v1.pdf
SUCCESS
Pages: 10
Elements: 112
Figures: 3
Tables: 3
Saved: ..\data\processed_papers\doc_538d7233661b.json

Processing: 2605.23953v1.pdf
SUCCESS
Pages: 13
Elements: 159
Figures: 1
Tables: 3
Saved: ..\data\processed_papers\doc_d34c8b40ce9b.json

Processing: 2605.23959v1.pdf
SUCCESS
Pages: 19
Elements: 246
Figures: 6
Tables: 8
Saved: ..\data\processed_papers\doc_9499a9575556.json

Processing: 2605.25894v1.pdf
SUCCESS
Pages: 6
Elements: 100
Figures: 5
Tables: 1
Saved: ..\data\processed_papers\doc_ac6daafe7996.json

Processing: 2605.27848v1.pdf
SUCCESS
Pages: 9
Elements: 145
Figures: 11
Tables: 8
Saved: ..\data\processed_papers\doc_6a8ad8a50fb1.json

Processing: 2605.27945v1.pdf
SUCCESS
Pages: 11
Elements: 271
Figures: 21
Tables: 20
Saved: ..\data\processed_papers\doc_83ca7df91b16.json

Processing: 2605.27977v1.pdf
SUCCESS
Pages: 24
Elements: 177
Figures: 10
Tables: 10
Saved: ..\data\processed_pape

In [14]:
import json
from pathlib import Path

# JSON file
json_path = Path("../data/processed_papers/doc_e37d68152c78.json")

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("JSON:", json_path)
print("Document ID:", data.get("document_id"))
print("Total elements:", len(data.get("elements", [])))

print("\n" + "=" * 80)
print("FIGURE VERIFICATION")
print("=" * 80)

figures = [
    element for element in data["elements"]
    if element.get("type") == "figure"
]

print(f"\nTotal figures found: {len(figures)}")

for fig in figures:
    image_path = fig.get("image_path")

    # JSON stores relative path
    if image_path:
        image_file = Path("../data/processed_papers") / image_path
        image_exists = image_file.exists()
    else:
        image_file = None
        image_exists = False

    print("\n" + "-" * 80)
    print("Figure ID       :", fig.get("figure_id"))
    print("Page            :", fig.get("page"))
    print("Section         :", fig.get("section_path"))
    print("Image path      :", image_path)
    print("Image exists?   :", image_exists)
    print("Caption         :", fig.get("caption"))
    print("Surrounding text:", fig.get("surrounding_text"))
    print("Related elements:", fig.get("related_element_ids"))

JSON: ..\data\processed_papers\doc_e37d68152c78.json
Document ID: doc_e37d68152c78
Total elements: 291

FIGURE VERIFICATION

Total figures found: 5

--------------------------------------------------------------------------------
Figure ID       : figure_0001
Page            : 16
Section         : ['3.1 Finite-difference schemes for the SDE and PDE with linear signature representations']
Image path      : images\doc_e37d68152c78\figure_0001.png
Image exists?   : True
Caption         : Figure 1: Accuracy of linear signature representations (˜v, ˜I) for truncation levels N = 1 to 5 under the OU process and mGBM: Each box spans from the first quartile (25%) to the third quartile (75%) of path-wise MAEs ǫ. The whisker boundaries denote the maximum and minimum ǫ. The overall MAEs E are marked out by the dashed (or dotted) lines. The horizontal axis uses a logarithmic scale, and the results are based on the Monte Carlo method with path number M = 10, 000 and the number of time intervals J = 

## Final multimodal evidence QA

This verifies that figure images actually exist and that tables contain either markdown or structured data.

In [15]:
total_figures = 0
figures_with_images = 0
total_tables = 0
tables_with_content = 0

for doc in saved_documents:
    figures = [e for e in doc['elements'] if e['type'] == 'figure']
    tables = [e for e in doc['elements'] if e['type'] == 'table']

    total_figures += len(figures)
    total_tables += len(tables)

    image_count = sum(
        bool(e.get('image_path'))
        and (PROCESSED_FOLDER / e['image_path']).exists()
        for e in figures
    )
    table_count = sum(
        bool(e.get('table_markdown') or e.get('table_data'))
        for e in tables
    )

    figures_with_images += image_count
    tables_with_content += table_count

    print(
        f"{doc['source_file']} | "
        f"figures: {len(figures)} ({image_count} images) | "
        f"tables: {len(tables)} ({table_count} with content)"
    )

print('\nTOTAL FIGURES:', total_figures)
print('FIGURES WITH IMAGES:', figures_with_images)
print('TOTAL TABLES:', total_tables)
print('TABLES WITH CONTENT:', tables_with_content)

2508.15237v2 (1).pdf | figures: 5 (5 images) | tables: 3 (3 with content)
2605.21504v1.pdf | figures: 3 (3 images) | tables: 3 (3 with content)
2605.23953v1.pdf | figures: 1 (1 images) | tables: 3 (3 with content)
2605.23959v1.pdf | figures: 6 (6 images) | tables: 8 (8 with content)
2605.25894v1.pdf | figures: 5 (5 images) | tables: 1 (1 with content)
2605.27848v1.pdf | figures: 11 (11 images) | tables: 8 (8 with content)
2605.27945v1.pdf | figures: 21 (21 images) | tables: 20 (20 with content)
2605.27977v1.pdf | figures: 10 (10 images) | tables: 10 (10 with content)
2605.28853v1.pdf | figures: 4 (4 images) | tables: 1 (1 with content)
2605.29413v1.pdf | figures: 1 (1 images) | tables: 11 (11 with content)
ssrn-5231379.pdf | figures: 10 (10 images) | tables: 0 (0 with content)

TOTAL FIGURES: 77
FIGURES WITH IMAGES: 77
TOTAL TABLES: 68
TABLES WITH CONTENT: 68


## Downstream contract for Notebook 02

Notebook 02 must **not discard multimodal evidence metadata**.

Every chunk should retain at minimum:

`chunk_id → document_id → element_ids → section_path → page(s) → evidence_type → image/table references`

This enables later retrieval of ordinary text, tables, and figures and allows the final RAG answer to trace back to the original PDF evidence.